# Phase 1: ShareGPT Baseline Analysis
## Comparing vLLM and QLM on Conversational Workloads

This notebook analyzes the results from Phase 1 experiments (E1.1-E1.12).

**Experiments:**
- E1.1-E1.2: Baseline (2 rps, mixed prompts)
- E1.3-E1.4: High rate (5 rps)
- E1.5-E1.6: Bursty traffic
- E1.7-E1.8: Multi-user (8 concurrent)
- E1.9-E1.10: Short prompts
- E1.11-E1.12: Long prompts

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Paths
RESULTS_DIR = Path("../results/phase1")
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

: 

## 1. Load Experiment Results

In [ ]:
def load_metrics(filename):
    """Load metrics JSON file."""
    path = RESULTS_DIR / filename
    if not path.exists():
        print(f"Warning: {filename} not found")
        return None
    with open(path) as f:
        return json.load(f)

# Load all experiments
experiments = {
    "E1.1": {"name": "vLLM Baseline", "file": "vllm_baseline_mixed.json", "system": "vLLM"},
    "E1.2": {"name": "QLM Baseline", "file": "qlm_baseline_mixed.json", "system": "QLM"},
    "E1.3": {"name": "vLLM High Rate", "file": "vllm_high_rate.json", "system": "vLLM"},
    "E1.4": {"name": "QLM High Rate", "file": "qlm_high_rate.json", "system": "QLM"},
    "E1.5": {"name": "vLLM Bursty", "file": "vllm_bursty.json", "system": "vLLM"},
    "E1.6": {"name": "QLM Bursty", "file": "qlm_bursty.json", "system": "QLM"},
    "E1.7": {"name": "vLLM Multi-User", "file": "vllm_multiuser.json", "system": "vLLM"},
    "E1.8": {"name": "QLM Multi-User", "file": "qlm_multiuser.json", "system": "QLM"},
    "E1.9": {"name": "vLLM Short", "file": "vllm_short_prompts.json", "system": "vLLM"},
    "E1.10": {"name": "QLM Short", "file": "qlm_short_prompts.json", "system": "QLM"},
    "E1.11": {"name": "vLLM Long", "file": "vllm_long_prompts.json", "system": "vLLM"},
    "E1.12": {"name": "QLM Long", "file": "qlm_long_prompts.json", "system": "QLM"},
}

# Load all data
for exp_id, exp in experiments.items():
    exp["data"] = load_metrics(exp["file"])

# Check which experiments loaded successfully
loaded = [exp_id for exp_id, exp in experiments.items() if exp["data"] is not None]
print(f"Loaded {len(loaded)}/{len(experiments)} experiments: {loaded}")

## 2. Summary Statistics

In [ ]:
# Create summary table
summary_data = []
for exp_id, exp in experiments.items():
    if exp["data"] is None:
        continue
    summary = exp["data"].get("summary", {})
    summary_data.append({
        "Experiment": exp_id,
        "Name": exp["name"],
        "System": exp["system"],
        "Requests": summary.get("num_requests_dispatched", 0),
        "Sched Delay (ms) - Mean": round(summary.get("scheduling_delay_ms_mean", 0), 2),
        "Sched Delay (ms) - P50": round(summary.get("scheduling_delay_ms_p50", 0), 2),
        "Sched Delay (ms) - P99": round(summary.get("scheduling_delay_ms_p99", 0), 2),
        "Queue Length - Mean": round(summary.get("queue_length_mean", 0), 2),
        "Queue Length - Max": summary.get("queue_length_max", 0),
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== Summary Statistics ===")
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv(PLOTS_DIR / "summary_statistics.csv", index=False)
print(f"\nSaved to {PLOTS_DIR / 'summary_statistics.csv'}")

## 3. Plot 1: Scheduling Delay Comparison (Baseline)

In [ ]:
# Compare E1.1 (vLLM) vs E1.2 (QLM) - Baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract scheduling delays
vllm_delays = [d["delay_ms"] for d in experiments["E1.1"]["data"]["scheduling_delays"]]
qlm_delays = [d["delay_ms"] for d in experiments["E1.2"]["data"]["scheduling_delays"]]

# Plot 1: CDF
ax = axes[0]
for delays, label, color in [(vllm_delays, "vLLM", "blue"), (qlm_delays, "QLM", "orange")]:
    sorted_delays = np.sort(delays)
    cdf = np.arange(1, len(sorted_delays) + 1) / len(sorted_delays)
    ax.plot(sorted_delays, cdf, label=label, linewidth=2, color=color)
ax.set_xlabel("Scheduling Delay (ms)")
ax.set_ylabel("CDF")
ax.set_title("Scheduling Delay CDF (Baseline: 2 rps, Mixed Prompts)")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Box plot
ax = axes[1]
ax.boxplot([vllm_delays, qlm_delays], labels=["vLLM", "QLM"])
ax.set_ylabel("Scheduling Delay (ms)")
ax.set_title("Scheduling Delay Distribution")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "plot1_scheduling_delay_baseline.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {PLOTS_DIR / 'plot1_scheduling_delay_baseline.png'}")

## 4. Plot 2: Queue Length Over Time

In [ ]:
# Compare queue dynamics for baseline experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

comparisons = [
    ("E1.1", "E1.2", "Baseline (2 rps)", axes[0, 0]),
    ("E1.3", "E1.4", "High Rate (5 rps)", axes[0, 1]),
    ("E1.5", "E1.6", "Bursty Traffic", axes[1, 0]),
    ("E1.7", "E1.8", "Multi-User (8 users)", axes[1, 1]),
]

for vllm_exp, qlm_exp, title, ax in comparisons:
    vllm_data = experiments[vllm_exp]["data"]
    qlm_data = experiments[qlm_exp]["data"]
    
    if vllm_data is None or qlm_data is None:
        continue
    
    # Extract queue length samples
    vllm_queue = vllm_data["queue_length_samples"]
    qlm_queue = qlm_data["queue_length_samples"]
    
    # Normalize time to start at 0
    vllm_t0 = vllm_queue[0]["t"] if vllm_queue else 0
    qlm_t0 = qlm_queue[0]["t"] if qlm_queue else 0
    
    vllm_times = [s["t"] - vllm_t0 for s in vllm_queue]
    vllm_lengths = [s["length"] for s in vllm_queue]
    
    qlm_times = [s["t"] - qlm_t0 for s in qlm_queue]
    qlm_lengths = [s["length"] for s in qlm_queue]
    
    ax.plot(vllm_times, vllm_lengths, label="vLLM", alpha=0.7, linewidth=1.5, color="blue")
    ax.plot(qlm_times, qlm_lengths, label="QLM", alpha=0.7, linewidth=1.5, color="orange")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Queue Length")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "plot2_queue_length_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {PLOTS_DIR / 'plot2_queue_length_over_time.png'}")

## 5. Plot 3: Throughput vs Load

In [ ]:
# Compare throughput (requests/sec) across different loads
load_experiments = [
    ("E1.1", "E1.2", "2 rps"),
    ("E1.3", "E1.4", "5 rps"),
]

vllm_throughputs = []
qlm_throughputs = []
load_labels = []

for vllm_exp, qlm_exp, label in load_experiments:
    vllm_data = experiments[vllm_exp]["data"]
    qlm_data = experiments[qlm_exp]["data"]
    
    if vllm_data is None or qlm_data is None:
        continue
    
    # Calculate throughput (requests / duration)
    vllm_duration = vllm_data.get("duration_sec", 60)
    qlm_duration = qlm_data.get("duration_sec", 60)
    
    vllm_requests = vllm_data["summary"]["num_requests_dispatched"]
    qlm_requests = qlm_data["summary"]["num_requests_dispatched"]
    
    vllm_throughputs.append(vllm_requests / vllm_duration)
    qlm_throughputs.append(qlm_requests / qlm_duration)
    load_labels.append(label)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(load_labels))
width = 0.35

ax.bar(x - width/2, vllm_throughputs, width, label="vLLM", color="blue", alpha=0.7)
ax.bar(x + width/2, qlm_throughputs, width, label="QLM", color="orange", alpha=0.7)

ax.set_xlabel("Arrival Rate")
ax.set_ylabel("Throughput (requests/sec)")
ax.set_title("Throughput vs Load (ShareGPT Baseline)")
ax.set_xticks(x)
ax.set_xticklabels(load_labels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "plot3_throughput_vs_load.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {PLOTS_DIR / 'plot3_throughput_vs_load.png'}")

## 6. Plot 4: Prompt Length Sensitivity

In [ ]:
# Compare performance on short vs long prompts
prompt_experiments = [
    ("E1.9", "E1.10", "Short (≤200 chars)"),
    ("E1.1", "E1.2", "Mixed"),
    ("E1.11", "E1.12", "Long (>1000 chars)"),
]

metrics_to_plot = [
    ("scheduling_delay_ms_mean", "Mean Scheduling Delay (ms)"),
    ("scheduling_delay_ms_p99", "P99 Scheduling Delay (ms)"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (metric_key, metric_label) in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    vllm_values = []
    qlm_values = []
    prompt_labels = []
    
    for vllm_exp, qlm_exp, label in prompt_experiments:
        vllm_data = experiments[vllm_exp]["data"]
        qlm_data = experiments[qlm_exp]["data"]
        
        if vllm_data is None or qlm_data is None:
            continue
        
        vllm_values.append(vllm_data["summary"].get(metric_key, 0))
        qlm_values.append(qlm_data["summary"].get(metric_key, 0))
        prompt_labels.append(label)
    
    x = np.arange(len(prompt_labels))
    width = 0.35
    
    ax.bar(x - width/2, vllm_values, width, label="vLLM", color="blue", alpha=0.7)
    ax.bar(x + width/2, qlm_values, width, label="QLM", color="orange", alpha=0.7)
    
    ax.set_xlabel("Prompt Length Category")
    ax.set_ylabel(metric_label)
    ax.set_title(f"{metric_label} by Prompt Length")
    ax.set_xticks(x)
    ax.set_xticklabels(prompt_labels, rotation=15, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "plot4_prompt_length_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {PLOTS_DIR / 'plot4_prompt_length_sensitivity.png'}")

## 7. Plot 5: Queue Dynamics Summary

In [ ]:
# Summary of queue length statistics across all experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prepare data
exp_names = []
vllm_mean_queue = []
qlm_mean_queue = []
vllm_max_queue = []
qlm_max_queue = []

pairs = [("E1.1", "E1.2", "Baseline"),
         ("E1.3", "E1.4", "High Rate"),
         ("E1.5", "E1.6", "Bursty"),
         ("E1.7", "E1.8", "Multi-User")]

for vllm_exp, qlm_exp, label in pairs:
    vllm_data = experiments[vllm_exp]["data"]
    qlm_data = experiments[qlm_exp]["data"]
    
    if vllm_data is None or qlm_data is None:
        continue
    
    exp_names.append(label)
    vllm_mean_queue.append(vllm_data["summary"].get("queue_length_mean", 0))
    qlm_mean_queue.append(qlm_data["summary"].get("queue_length_mean", 0))
    vllm_max_queue.append(vllm_data["summary"].get("queue_length_max", 0))
    qlm_max_queue.append(qlm_data["summary"].get("queue_length_max", 0))

x = np.arange(len(exp_names))
width = 0.35

# Plot 1: Mean queue length
ax = axes[0]
ax.bar(x - width/2, vllm_mean_queue, width, label="vLLM", color="blue", alpha=0.7)
ax.bar(x + width/2, qlm_mean_queue, width, label="QLM", color="orange", alpha=0.7)
ax.set_xlabel("Experiment")
ax.set_ylabel("Mean Queue Length")
ax.set_title("Mean Queue Length Comparison")
ax.set_xticks(x)
ax.set_xticklabels(exp_names, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Max queue length
ax = axes[1]
ax.bar(x - width/2, vllm_max_queue, width, label="vLLM", color="blue", alpha=0.7)
ax.bar(x + width/2, qlm_max_queue, width, label="QLM", color="orange", alpha=0.7)
ax.set_xlabel("Experiment")
ax.set_ylabel("Max Queue Length")
ax.set_title("Max Queue Length Comparison")
ax.set_xticks(x)
ax.set_xticklabels(exp_names, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "plot5_queue_dynamics_summary.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {PLOTS_DIR / 'plot5_queue_dynamics_summary.png'}")

## 8. Key Findings Summary

In [ ]:
print("="*60)
print("KEY FINDINGS: Phase 1 (ShareGPT Baseline)")
print("="*60)
print()

# Calculate improvements
def calculate_improvement(vllm_exp, qlm_exp, metric):
    vllm_val = experiments[vllm_exp]["data"]["summary"].get(metric, 0)
    qlm_val = experiments[qlm_exp]["data"]["summary"].get(metric, 0)
    if vllm_val == 0:
        return 0
    return ((vllm_val - qlm_val) / vllm_val) * 100

# Baseline comparison
baseline_delay_improvement = calculate_improvement("E1.1", "E1.2", "scheduling_delay_ms_mean")
baseline_p99_improvement = calculate_improvement("E1.1", "E1.2", "scheduling_delay_ms_p99")
baseline_queue_improvement = calculate_improvement("E1.1", "E1.2", "queue_length_mean")

print(f"1. BASELINE (2 rps, mixed prompts):")
print(f"   - QLM reduces mean scheduling delay by {baseline_delay_improvement:.1f}%")
print(f"   - QLM reduces P99 scheduling delay by {baseline_p99_improvement:.1f}%")
print(f"   - QLM reduces mean queue length by {baseline_queue_improvement:.1f}%")
print()

# High load comparison
highload_delay_improvement = calculate_improvement("E1.3", "E1.4", "scheduling_delay_ms_mean")
print(f"2. HIGH LOAD (5 rps):")
print(f"   - QLM reduces mean scheduling delay by {highload_delay_improvement:.1f}%")
print()

# Bursty traffic
bursty_queue_improvement = calculate_improvement("E1.5", "E1.6", "queue_length_max")
print(f"3. BURSTY TRAFFIC:")
print(f"   - QLM reduces max queue length by {bursty_queue_improvement:.1f}%")
print()

# Prompt length sensitivity
print(f"4. PROMPT LENGTH SENSITIVITY:")
print(f"   - Short prompts: {calculate_improvement('E1.9', 'E1.10', 'scheduling_delay_ms_mean'):.1f}% delay reduction")
print(f"   - Long prompts: {calculate_improvement('E1.11', 'E1.12', 'scheduling_delay_ms_mean'):.1f}% delay reduction")
print()

print("="*60)
print("NEXT STEPS:")
print("="*60)
print("1. Add these findings to Overleaf report (Results section)")
print("2. Include 5 generated plots in report")
print("3. Write discussion of QLM's SLO-aware scheduling benefits")
print("4. Move to Phase 2: Agentic trace experiments")
print()

## 9. Export for Report

In [ ]:
# Export summary table as LaTeX
latex_table = summary_df.to_latex(index=False, float_format="%.2f")
with open(PLOTS_DIR / "summary_table.tex", "w") as f:
    f.write(latex_table)

print(f"Exported LaTeX table to {PLOTS_DIR / 'summary_table.tex'}")
print("\nAll plots saved to:", PLOTS_DIR.absolute())
print("\nReady for Overleaf!")